# PHSX 256 Topic 14: Numerical Function Minimization

---
## Why Minimization in Physics?

Many of the deepest principles in physics are *extremal principles*. The true path of a particle minimizes (or extremizes) the action. The equilibrium state of a thermodynamic system minimizes the free energy. The ground-state wavefunction minimizes the expectation value of the Hamiltonian. In experimental physics, fitting data to a model means minimizing a cost function such as chi-squared, $\chi^2$.

In practice, a **minimization problem** has the form:

$$x^* = \underset{x}{\mathrm{argmin}}\; f(x)$$

where $f(x)$ is called the **objective function** (or cost function), and $x^*$ is the value at which $f$ attains its minimum. We will almost always look for **local** minima — minima within some bracketed region — and we must be careful to distinguish them from **global** minima.

### Necessary and sufficient conditions

For a smooth function, a local minimum at $x^*$ requires:

$$f'(x^*) = 0 \qquad \text{(stationary point)}$$

$$f''(x^*) > 0 \qquad \text{(local minimum, not maximum or saddle point)}$$

Numerically, we can rarely guarantee we have a *global* minimum; we find a local one and use physical reasoning to judge whether it is the one we want.

In [ ]:
# Standard imports used throughout this notebook
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12,
    'lines.linewidth': 2,
})
print("Imports successful.")

---
## Brute-Force Grid Search

The simplest conceivable approach is to evaluate $f(x)$ at every point on a uniformly spaced grid and report the grid point with the smallest value. Despite its naivety, grid search is invaluable as a **first pass**: it reveals the global landscape, locates all local minima, and provides a bracket to feed into faster algorithms.

### Algorithm

Given an interval $[a, b]$ divided into $N$ equally spaced points,

$$x_i = a + i\,\Delta x, \qquad \Delta x = \frac{b-a}{N-1}, \qquad i = 0, 1, \dots, N-1$$

we compute $f(x_i)$ for all $i$ and find $i^* = \mathrm{argmin}_i\, f(x_i)$.

### Accuracy and cost

The resolution of the result is limited by the grid spacing $\Delta x$. To halve the error, you must double $N$ — the cost scales as $O(N)$ function evaluations. For 1-D problems this is perfectly acceptable; in $d$ dimensions the cost grows as $O(N^d)$, making grid search impractical for high-dimensional spaces (the **curse of dimensionality**).

### Physics example 1 — Lennard-Jones potential

The Lennard-Jones (LJ) potential models the interaction energy between two neutral atoms:

$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

The $r^{-12}$ term represents short-range Pauli repulsion; the $r^{-6}$ term represents long-range van der Waals attraction. The equilibrium separation $r_0$ minimizes $U(r)$.

In [ ]:
# Brute-Force Grid Search: Lennard-Jones Potential

def lennard_jones(r, eps=1.0, sigma=1.0):
    # Lennard-Jones pair potential in reduced units (eps=sigma=1)
    return 4 * eps * ((sigma / r)**12 - (sigma / r)**6)

# Build uniform grid (avoid r -> 0 divergence)
r_grid = np.linspace(0.85, 3.0, 5000)
U_grid = lennard_jones(r_grid)

idx_min    = np.argmin(U_grid)
r_min_grid = r_grid[idx_min]
U_min_grid = U_grid[idx_min]

# Analytic minimum: r0 = 2^(1/6) * sigma
r_min_exact = 2**(1/6)
U_min_exact = lennard_jones(r_min_exact)

print(f"Grid search minimum:  r = {r_min_grid:.6f},  U = {U_min_grid:.6f}")
print(f"Analytic minimum:     r = {r_min_exact:.6f},  U = {U_min_exact:.6f}")
print(f"Error in r:           dr = {abs(r_min_grid - r_min_exact):.2e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_grid, U_grid, 'steelblue', label=r'$U(r)$ — Lennard-Jones')
ax.axhline(0, color='gray', lw=1, ls='--')
ax.plot(r_min_grid, U_min_grid, 'ro', ms=10, zorder=5,
        label=f'Grid minimum  $r^* = {r_min_grid:.4f}$')
ax.plot(r_min_exact, U_min_exact, 'g^', ms=10, zorder=5,
        label=f'Exact minimum  $r_0 = 2^{{1/6}} = {r_min_exact:.4f}$')
ax.set_xlabel(r'Separation $r\,/\,\sigma$')
ax.set_ylabel(r'Potential energy $U\,/\,\varepsilon$')
ax.set_title('Lennard-Jones Potential — Brute-Force Grid Search')
ax.set_ylim(-1.5, 2)
ax.legend()
plt.tight_layout()
plt.show()

### Physics example 2 — Pendulum: finding the amplitude that matches a target period

For a simple pendulum of length $L$, the exact period for initial amplitude $\theta_0$ is:

$$T(\theta_0) = 4\sqrt{\frac{L}{g}}\,K\!\left(\sin\frac{\theta_0}{2}\right)$$

where $K(k)$ is the complete elliptic integral of the first kind. In the small-angle limit $T \approx 2\pi\sqrt{L/g}$. For large angles the period is longer.

We pose the inverse problem: given a target period $T_{\rm target}$, find the amplitude $\theta_0$ that achieves it. This is a minimization problem — minimize the cost function $C(\theta_0) = |T(\theta_0) - T_{\rm target}|$.

In [ ]:
# Brute-Force Grid Search: Pendulum Period

from scipy.special import ellipk

def pendulum_period(theta0_deg, L=1.0, g=9.81):
    # Exact pendulum period via elliptic integral
    theta0 = np.radians(theta0_deg)
    k = np.clip(np.sin(theta0 / 2), 0, 0.9999)
    return 4 * np.sqrt(L / g) * ellipk(k**2)

T_small_angle = 2 * np.pi * np.sqrt(1.0 / 9.81)   # L=1 m
T_target = 2.10   # seconds — larger than small-angle period, so a large amplitude is needed

theta_grid = np.linspace(1, 175, 2000)
T_grid     = np.array([pendulum_period(th) for th in theta_grid])
cost_grid  = np.abs(T_grid - T_target)

idx_min   = np.argmin(cost_grid)
theta_min = theta_grid[idx_min]

print(f"Small-angle period (L=1 m): T0 = {T_small_angle:.4f} s")
print(f"Target period: T_target = {T_target:.4f} s")
print(f"Grid result: theta0 = {theta_min:.2f} deg  ->  T = {pendulum_period(theta_min):.4f} s")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(theta_grid, T_grid, 'darkorange')
axes[0].axhline(T_target, color='red', ls='--', label=f'$T_{{\\rm target}}={T_target}$ s')
axes[0].axhline(T_small_angle, color='gray', ls=':', label=f'Small-angle $T_0={T_small_angle:.3f}$ s')
axes[0].plot(theta_min, pendulum_period(theta_min), 'bo', ms=10,
             label=f'Grid minimum  $\\theta_0={theta_min:.1f}°$')
axes[0].set_xlabel(r'Amplitude $\theta_0$ (degrees)')
axes[0].set_ylabel('Period $T$ (s)')
axes[0].set_title('Pendulum Period vs. Amplitude')
axes[0].legend(fontsize=10)

axes[1].plot(theta_grid, cost_grid, 'purple')
axes[1].plot(theta_min, cost_grid[idx_min], 'ro', ms=10, label='Grid minimum')
axes[1].set_xlabel(r'Amplitude $\theta_0$ (degrees)')
axes[1].set_ylabel(r'$|T(\theta_0) - T_{\rm target}|$ (s)')
axes[1].set_title('Cost Function')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Bracketing a Minimum

Before applying a fast refinement algorithm, we need to **bracket** the minimum — find three points $a < b < c$ such that $f(b) < f(a)$ and $f(b) < f(c)$. By continuity, a local minimum must exist somewhere in $(a, c)$.

A grid search naturally provides a bracket: once we find the grid index $i^*$ of the smallest value, the bracket is $(x_{i^*-1},\, x_{i^*},\, x_{i^*+1})$. The **width** of the bracket is our starting uncertainty; the refinement algorithm then shrinks it efficiently.

---
## Golden-Section Search

Golden-section search is the 1-D minimization analogue of binary search. Given a bracket $[a, c]$ with a known interior minimum, the algorithm evaluates one new point per iteration and reduces the bracket by a fixed factor — determined by the **golden ratio**.

### The golden ratio and why it is optimal

Define the golden ratio $\varphi = \frac{1+\sqrt{5}}{2} \approx 1.618$. Its key property is:

$$\varphi^2 = \varphi + 1 \qquad \Longleftrightarrow \qquad \frac{1}{\varphi} = \varphi - 1 \approx 0.618$$

The algorithm places interior test points at fractions $1 - 1/\varphi \approx 0.382$ and $1/\varphi \approx 0.618$ of the interval. After each function evaluation, the bracket shrinks by a factor of $1/\varphi \approx 0.618$, regardless of which side is discarded. It can be shown that no 1-D algorithm using only function values (no derivatives) can guarantee a better reduction ratio per evaluation. The method is therefore **optimal** among derivative-free methods.

### Algorithm

Given bracket $[a, c]$:

1. Place two interior points:
$$x_1 = a + \left(1 - \frac{1}{\varphi}\right)(c - a), \qquad x_2 = a + \frac{1}{\varphi}(c - a)$$

2. Evaluate $f_1 = f(x_1)$, $f_2 = f(x_2)$.

3. If $f_1 < f_2$: discard the right portion — set $c \leftarrow x_2$ and reuse $x_1$.

4. Else: discard the left portion — set $a \leftarrow x_1$ and reuse $x_2$.

5. Repeat until $c - a < \varepsilon_{\rm tol}$.

After $n$ iterations the bracket width is $(c_0 - a_0)/\varphi^n$. To achieve tolerance $\varepsilon$:

$$n = \frac{\ln\!\bigl[(c_0 - a_0)/\varepsilon\bigr]}{\ln \varphi}$$

### Convergence

Golden-section search converges **linearly** — each iteration reduces the bracket by the same constant factor $1/\varphi$. This is slower than Newton's method (which converges quadratically) but it is **guaranteed** to converge and requires no derivative information.

In [ ]:
# Golden-Section Search Implementation

def golden_section_search(f, a, c, tol=1e-10, maxiter=300):
    # Minimize f on [a, c] using golden-section search.
    # Returns (x_min, f_min, list_of_bracket_widths).
    phi    = (1 + np.sqrt(5)) / 2
    resphi = 2 - phi          # = 1 - 1/phi ~= 0.382

    x1 = a + resphi * (c - a)
    x2 = c - resphi * (c - a)
    f1, f2 = f(x1), f(x2)

    history = [c - a]

    for _ in range(maxiter):
        if f1 < f2:
            c, x2, f2 = x2, x1, f1
            x1 = a + resphi * (c - a)
            f1 = f(x1)
        else:
            a, x1, f1 = x1, x2, f2
            x2 = c - resphi * (c - a)
            f2 = f(x2)
        history.append(c - a)
        if c - a < tol:
            break

    x_min = (a + c) / 2
    return x_min, f(x_min), history


# Quick sanity check: minimize x^2 on [-3, 5], minimum at x=0
x_test, f_test, _ = golden_section_search(lambda x: x**2, -3, 5)
print(f"Test (x^2):  x* = {x_test:.2e}   (should be 0.00)")

### Physics example 3 — Spring-gravity equilibrium

Consider a mass $m$ hanging from a spring (spring constant $k$, natural length $L_0$). If the spring makes angle $\theta$ with the vertical, the total potential energy as a function of the extension $x$ beyond the natural length is:

$$U(x) = \frac{1}{2}k x^2 - mg(L_0 + x)\cos\theta$$

The equilibrium extension minimizes $U(x)$. Setting $dU/dx = 0$ gives the analytic result $x^* = mg\cos\theta / k$, which we verify numerically.

In [ ]:
# Golden-Section: Spring-Gravity Equilibrium

m, g_acc, k, L0, theta_deg = 2.0, 9.81, 50.0, 0.5, 30.0

def spring_gravity_PE(x):
    theta_r = np.radians(theta_deg)
    return 0.5 * k * x**2 - m * g_acc * (L0 + x) * np.cos(theta_r)

x_analytic = m * g_acc * np.cos(np.radians(theta_deg)) / k
x_gs, U_gs, history_spring = golden_section_search(spring_gravity_PE, -0.5, 1.5)

print(f"Golden-section:  x* = {x_gs:.8f} m")
print(f"Analytic:        x* = {x_analytic:.8f} m")
print(f"Error: {abs(x_gs - x_analytic):.2e} m  after {len(history_spring)} iterations")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_arr = np.linspace(-0.2, 1.0, 400)
U_arr = spring_gravity_PE(x_arr)

axes[0].plot(x_arr, U_arr, 'steelblue', label=r'$U(x)$')
axes[0].plot(x_gs, U_gs, 'ro', ms=10, zorder=5,
             label=f'GS minimum  $x^*={x_gs:.4f}$ m')
axes[0].axvline(x_analytic, color='green', ls='--', lw=1.5,
                label=f'Analytic  $x_{{\\rm eq}}={x_analytic:.4f}$ m')
axes[0].set_xlabel('Extension $x$ (m)')
axes[0].set_ylabel('Potential energy $U$ (J)')
axes[0].set_title('Spring-Gravity Potential Energy')
axes[0].legend(fontsize=10)

axes[1].semilogy(range(len(history_spring)), history_spring, 'darkorange', marker='o', ms=4)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Bracket width (m)')
axes[1].set_title('Convergence of Golden-Section Search\n(linear on log scale)')

plt.tight_layout()
plt.show()

### Physics example 4 — Fermat's principle and Snell's law

Fermat's principle states that light travels along the path of **least time**. Consider a ray traveling from point $A = (0, y_A)$ in medium 1 (speed $v_1$) to point $B = (d, -y_B)$ in medium 2 (speed $v_2$), crossing the interface at horizontal coordinate $x$. The total travel time is:

$$T(x) = \frac{\sqrt{x^2 + y_A^2}}{v_1} + \frac{\sqrt{(d-x)^2 + y_B^2}}{v_2}$$

Minimizing $T(x)$ is equivalent to deriving Snell's law: $\sin\theta_1/v_1 = \sin\theta_2/v_2$.

In [ ]:
# Golden-Section: Fermat's Principle => Snell's Law

yA, yB, d   = 2.0, 1.5, 3.0
v1, v2      = 1.0, 0.6          # v2 < v1: light bends toward normal (like glass)

def travel_time(x):
    t1 = np.sqrt(x**2 + yA**2) / v1
    t2 = np.sqrt((d - x)**2 + yB**2) / v2
    return t1 + t2

x_opt, T_opt, hist_fermat = golden_section_search(travel_time, 0.01, d - 0.01)

sin1 = x_opt / np.sqrt(x_opt**2 + yA**2)
sin2 = (d - x_opt) / np.sqrt((d - x_opt)**2 + yB**2)
print(f"Optimal crossing:  x* = {x_opt:.6f} m")
print(f"sin(theta1)/v1 = {sin1/v1:.8f}")
print(f"sin(theta2)/v2 = {sin2/v2:.8f}")
print(f"Snell's law satisfied: {np.isclose(sin1/v1, sin2/v2, atol=1e-7)}")

theta1_deg = np.degrees(np.arcsin(sin1))
theta2_deg = np.degrees(np.arcsin(sin2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_scan = np.linspace(0.05, d - 0.05, 500)
axes[0].plot(x_scan, travel_time(x_scan), 'steelblue')
axes[0].plot(x_opt, T_opt, 'ro', ms=10, zorder=5, label=f'Min at $x^*={x_opt:.3f}$ m')
axes[0].set_xlabel('Crossing point $x$ (m)')
axes[0].set_ylabel('Travel time $T$ (s)')
axes[0].set_title("Fermat's Principle — Travel Time")
axes[0].legend()

ax2 = axes[1]
ax2.fill_between([0, d], [-3, -3], [0, 0], alpha=0.15, color='cyan',
                 label=f'Medium 2  ($v_2={v2}$)')
ax2.fill_between([0, d], [0, 0], [3, 3], alpha=0.10, color='yellow',
                 label=f'Medium 1  ($v_1={v1}$)')
ax2.plot([0, x_opt], [yA, 0], 'b-o', lw=2.5, label='Incident ray')
ax2.plot([x_opt, d], [0, -yB], 'r-o', lw=2.5, label='Refracted ray')
ax2.axhline(0, color='k', lw=1.5, ls='--')
ax2.set_title(f'Ray Diagram:  $\\theta_1={theta1_deg:.1f}°$,  $\\theta_2={theta2_deg:.1f}°$')
ax2.set_xlabel('$x$ (m)')
ax2.set_ylabel('$y$ (m)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## Parabolic Interpolation

Golden-section search discards information: it uses only the *ordering* of function values, not their magnitudes. **Parabolic interpolation** exploits the magnitudes — it fits a parabola through three known points and jumps directly to the parabola's minimum.

### Derivation

Given three points $x_1 < x_2 < x_3$ with function values $f_1, f_2, f_3$, the unique parabola through these points has its vertex at:

$$x^* = x_2 - \frac{1}{2}\,\frac{(x_2 - x_1)^2(f_2 - f_3) - (x_2 - x_3)^2(f_2 - f_1)}{(x_2 - x_1)(f_2 - f_3) - (x_2 - x_3)(f_2 - f_1)}$$

This is derived by writing the Lagrange interpolating polynomial through the three points and setting its derivative to zero.

### When it works — and when it fails

Parabolic interpolation converges **superlinearly** (faster than linear, slower than quadratic) when the function is smooth and well approximated by a parabola near the minimum. However, it can produce wild extrapolations when:
- The three points span a non-convex region,
- The denominator is near zero (nearly flat curvature),
- There is significant noise in the function values.

In practice, parabolic steps are combined with golden-section fallback steps to get the best of both worlds.

### Convergence rate

Parabolic interpolation achieves superlinear convergence with rate $\approx 1.324$ — meaning the number of correct digits grows by a factor of $1.324$ per iteration.

In [ ]:
# Parabolic Interpolation Implementation

def parabolic_vertex(x1, x2, x3, f1, f2, f3):
    # Return the x-coordinate of the vertex of the parabola through
    # (x1,f1), (x2,f2), (x3,f3). Returns None if the denominator is too small.
    denom = (x2 - x1) * (f2 - f3) - (x2 - x3) * (f2 - f1)
    numer = (x2 - x1)**2 * (f2 - f3) - (x2 - x3)**2 * (f2 - f1)
    if abs(denom) < 1e-14:
        return None
    return x2 - 0.5 * numer / denom


def parabolic_search(f, a, b, c, tol=1e-10, maxiter=200):
    # Minimize f in bracket (a, b, c) using parabolic interpolation.
    # Falls back to a golden-section step if the parabolic step leaves the bracket.
    # Returns (x_min, f_min, history_of_best_estimates).
    phi    = (1 + np.sqrt(5)) / 2
    resphi = 2 - phi

    fa, fb, fc = f(a), f(b), f(c)
    history = [b]

    for _ in range(maxiter):
        x_para = parabolic_vertex(a, b, c, fa, fb, fc)

        if x_para is not None and a < x_para < c:
            x_new, f_new = x_para, f(x_para)
        else:
            # Golden-section fallback
            if (b - a) > (c - b):
                x_new = b - resphi * (b - a)
            else:
                x_new = b + resphi * (c - b)
            f_new = f(x_new)

        # Update bracket to keep three points straddling the minimum
        if f_new < fb:
            if x_new < b:
                c, fc = b, fb
            else:
                a, fa = b, fb
            b, fb = x_new, f_new
        else:
            if x_new < b:
                a, fa = x_new, f_new
            else:
                c, fc = x_new, f_new

        history.append(b)
        if c - a < tol:
            break

    return b, fb, history

### Physics example 5 — Morse potential (diatomic molecule)

The **Morse potential** is a realistic model for the potential energy of a diatomic molecule:

$$V(r) = D_e\left(1 - e^{-a(r - r_e)}\right)^2$$

where $D_e$ is the dissociation energy, $r_e$ is the equilibrium bond length, and $a$ controls the width of the potential well. By construction, the minimum is at $r = r_e$. We find it numerically and compare parabolic interpolation with golden-section search.

In [ ]:
# Parabolic Interpolation vs. Golden-Section: Morse Potential

De, a_morse, re = 1.0, 1.5, 1.0   # reduced units

def morse(r):
    return De * (1 - np.exp(-a_morse * (r - re)))**2

a0, b0, c0 = 0.5, 0.9, 2.5   # initial bracket

x_gs,   f_gs,   hist_gs   = golden_section_search(morse, a0, c0)
x_para, f_para, hist_para = parabolic_search(morse, a0, b0, c0)

print(f"Golden-section:  r* = {x_gs:.10f}   iters = {len(hist_gs)}")
print(f"Parabolic:       r* = {x_para:.10f}   iters = {len(hist_para)}")
print(f"Exact:           r* = {re:.10f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

r_arr = np.linspace(0.3, 3.5, 500)
axes[0].plot(r_arr, morse(r_arr), 'steelblue', label=r'Morse $V(r)$')
axes[0].plot(x_gs,   f_gs,   'ro', ms=10, zorder=5, label=f'GS:  $r^*={x_gs:.5f}$')
axes[0].plot(x_para, f_para, 'g^', ms=10, zorder=5, label=f'Parabolic:  $r^*={x_para:.5f}$')
axes[0].set_xlabel(r'Bond length $r$')
axes[0].set_ylabel(r'$V(r)$')
axes[0].set_title('Morse Potential')
axes[0].legend()

# Convergence: distance to true minimum
err_gs   = [abs(e - re) for e in np.linspace(a0, c0, len(hist_gs))]  # approximate
err_gs2  = [abs(0.5*(a0+c0)/((1.618)**(i)) + re - re) for i in range(len(hist_gs))]  # schematic
axes[1].semilogy([abs(h - re) for h in hist_gs],   'b-o', ms=4,
                 label=f'Golden-section ({len(hist_gs)} iter)')
axes[1].semilogy([abs(h - re) for h in hist_para], 'g-^', ms=4,
                 label=f'Parabolic ({len(hist_para)} iter)')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('$|x_n - r_e|$')
axes[1].set_title('Convergence Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

### Physics example 6 — Angle of maximum projectile range

For a projectile launched at speed $v_0$ from height $h$ above flat ground (no air resistance), the horizontal range is:

$$R(\theta) = \frac{v_0 \cos\theta}{g}\left(v_0 \sin\theta + \sqrt{v_0^2 \sin^2\theta + 2gh}\right)$$

For $h = 0$ the maximum occurs at exactly $\theta = 45°$. For $h > 0$ the optimal angle is less than $45°$. We maximize $R$ by minimizing $-R(\theta)$.

In [ ]:
# Parabolic Interpolation: Optimal Projectile Launch Angle

v0_proj, g_acc2, h_launch = 20.0, 9.81, 5.0    # m/s, m/s^2, m

def minus_range(theta_deg):
    theta = np.radians(theta_deg)
    ct, st = np.cos(theta), np.sin(theta)
    R = (v0_proj * ct / g_acc2) * (v0_proj * st + np.sqrt((v0_proj * st)**2 + 2 * g_acc2 * h_launch))
    return -R

# Coarse grid search to locate bracket
theta_coarse = np.linspace(1, 89, 200)
idx_br = np.argmin([minus_range(t) for t in theta_coarse])
a_br = theta_coarse[max(0, idx_br - 3)]
c_br = theta_coarse[min(len(theta_coarse)-1, idx_br + 3)]
b_br = theta_coarse[idx_br]

x_opt2, _, hist_proj = parabolic_search(minus_range, a_br, b_br, c_br)
print(f"Optimal launch angle: theta* = {x_opt2:.4f} deg  (h={h_launch} m, v0={v0_proj} m/s)")
print(f"Maximum range:  R = {-minus_range(x_opt2):.4f} m")
print(f"For h=0 the answer is exactly 45 deg; h={h_launch} m shifts it to {x_opt2:.2f} deg")

theta_plot = np.linspace(1, 89, 400)
R_plot = np.array([-minus_range(t) for t in theta_plot])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(theta_plot, R_plot, 'steelblue')
ax.axvline(x_opt2, color='red', ls='--', label=f'$\\theta^*={x_opt2:.2f}°$ (parabolic)')
ax.axvline(45, color='gray', ls=':', lw=1.5, label='45° (h=0 limit)')
ax.set_xlabel('Launch angle $\\theta$ (degrees)')
ax.set_ylabel('Range $R$ (m)')
ax.set_title(f'Projectile Range  ($h={h_launch}$ m, $v_0={v0_proj}$ m/s)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Brent's Method

**Brent's method** (Richard Brent, 1973) is the gold standard for 1-D minimization without derivatives. It intelligently combines golden-section search and parabolic interpolation:

- It attempts a parabolic step.
- If the parabolic step lies within the bracket and would reduce the interval by more than a threshold, it is accepted.
- Otherwise, a golden-section step is taken as a safe fallback.

This prevents the pathological cases that can afflict pure parabolic interpolation, while retaining superlinear convergence in well-behaved regions and guaranteed linear convergence everywhere.

### Key properties compared

| Property | Golden-Section | Parabolic Interp. | Brent's Method |
|---|---|---|---|
| Derivative needed? | No | No | No |
| Guaranteed convergence? | Yes | No | Yes |
| Convergence rate | Linear | Superlinear | Superlinear |
| Worst-case behavior | $O(1/\varphi^n)$ | Can diverge | $O(1/\varphi^n)$ |

In practice, **always use Brent's method** (or its `scipy` implementation) unless you have a specific reason to do otherwise. `scipy.optimize.minimize_scalar` with `method='brent'` or `method='bounded'` uses Brent's method.

In [ ]:
# Brent's Method via scipy.optimize

from scipy.optimize import minimize_scalar

# Physics example: effective potential in orbital mechanics
# V_eff(r) = L^2/(2mr^2) - GMm/r   (centrifugal + gravitational)
# The minimum gives the circular orbit radius.

G_grav    = 6.674e-11   # m^3 kg^-1 s^-2
M_star    = 2e30        # kg (solar mass)
m_part    = 1.0         # kg (test particle)
L_orb     = 4e40        # kg m^2 / s (angular momentum)

def V_eff(r):
    centrifugal = L_orb**2 / (2 * m_part * r**2)
    gravity     = -G_grav * M_star * m_part / r
    return centrifugal + gravity

result_brent = minimize_scalar(V_eff, bounds=(1e9, 1e12), method='bounded')
r_eq = result_brent.x

# Analytic circular orbit: dV_eff/dr = 0  =>  r* = L^2 / (G M m^2)
r_analytic2 = L_orb**2 / (G_grav * M_star * m_part**2)

print(f"Brent's method:  r* = {r_eq:.6e} m")
print(f"Analytic:        r* = {r_analytic2:.6e} m")
print(f"Relative error: {abs(r_eq - r_analytic2)/r_analytic2:.2e}")

r_arr2 = np.linspace(1.5e10, 8e11, 1000)
V_arr2 = V_eff(r_arr2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_arr2 / 1e10, V_arr2 / 1e30, 'steelblue', label=r'$V_{\rm eff}(r)$')
ax.plot(r_eq / 1e10, result_brent.fun / 1e30, 'ro', ms=12, zorder=5,
        label=f"Brent minimum  $r^*={r_eq:.2e}$ m")
ax.axhline(0, color='gray', lw=1, ls='--')
ax.set_xlabel(r'$r$ ($\times 10^{10}$ m)')
ax.set_ylabel(r'$V_{\rm eff}$ ($\times 10^{30}$ J)')
ax.set_title("Effective Orbital Potential — Brent's Method (scipy)")
ax.legend()
plt.tight_layout()
plt.show()

### Physics example 7 — Wien's displacement law from Planck's spectrum

Planck's law for the spectral radiance of a blackbody at temperature $T$ is:

$$B(\nu, T) = \frac{2h\nu^3}{c^2} \cdot \frac{1}{e^{h\nu / k_B T} - 1}$$

The frequency of peak emission $\nu_{\rm max}$ satisfies Wien's displacement law: $\nu_{\rm max} = \alpha\, T$ where $\alpha \approx 5.879 \times 10^{10}$ Hz/K. We find $\nu_{\rm max}$ numerically by minimizing $-B(\nu, T)$.

In [ ]:
# Brent's Method: Wien's Displacement Law

h_pl    = 6.626e-34   # Planck constant (J s)
c_light = 3e8         # speed of light (m/s)
kB      = 1.381e-23   # Boltzmann constant (J/K)
alpha_wien = 5.879e10 # Hz/K (theoretical)

def minus_planck(nu, T):
    x = h_pl * nu / (kB * T)
    return -(2 * h_pl * nu**3 / c_light**2) / (np.exp(x) - 1)

T_values = [3000, 5778, 10000]   # cool star, Sun, hot star
colors   = ['firebrick', 'gold', 'cornflowerblue']
labels   = ['Cool star (3000 K)', 'Sun (5778 K)', 'Hot star (10000 K)']

fig, ax = plt.subplots(figsize=(9, 5))

for T, col, lbl in zip(T_values, colors, labels):
    nu_arr  = np.linspace(1e12, 2e15, 3000)
    B_arr   = -np.array([minus_planck(nu, T) for nu in nu_arr])

    res_w   = minimize_scalar(minus_planck, args=(T,),
                              bounds=(1e12, 2e15), method='bounded')
    nu_max  = res_w.x
    nu_wien = alpha_wien * T

    print(f"T = {T:5d} K:  nu_max(Brent) = {nu_max:.4e} Hz  |  "
          f"nu_max(Wien) = {nu_wien:.4e} Hz  |  "
          f"error = {abs(nu_max-nu_wien)/nu_wien:.2e}")

    ax.plot(nu_arr / 1e14, B_arr / B_arr.max(), color=col, label=lbl)
    ax.axvline(nu_max / 1e14, color=col, ls='--', alpha=0.7, lw=1.5)

ax.set_xlabel(r'Frequency $\nu$ ($\times 10^{14}$ Hz)')
ax.set_ylabel('Spectral radiance (normalized)')
ax.set_title("Planck Spectrum — Peak Frequencies via Brent's Method")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## Newton's Method for Minimization

If we have access to the **first and second derivatives** $f'(x)$ and $f''(x)$ — either analytically or via numerical finite differences — we can find the minimum much faster.

### Idea

Finding the minimum of $f$ means finding a zero of $f'$. Applying Newton's root-finding update to $f'(x) = 0$:

$$x_{n+1} = x_n - \frac{f'(x_n)}{f''(x_n)}$$

Geometrically: at each step, we fit a local parabola using $f'(x_n)$ and $f''(x_n)$ and jump directly to that parabola's vertex.

### Convergence

Newton's method converges **quadratically** near a simple minimum. If $e_n = x_n - x^*$ is the error at step $n$:

$$e_{n+1} \approx -\frac{f'''(x^*)}{2\,f''(x^*)}\,e_n^2$$

The number of correct decimal digits roughly **doubles** with every iteration — extremely fast once you are in the basin of convergence.

### Numerical derivatives

When an analytic derivative is unavailable, use finite differences:

$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h} \qquad \text{(central difference, error } O(h^2)\text{)}$$

$$f''(x) \approx \frac{f(x + h) - 2f(x) + f(x - h)}{h^2} \qquad \text{(error } O(h^2)\text{)}$$

The step size $h$ must balance truncation error ($\sim h^2$) and floating-point rounding ($\sim \varepsilon_{\rm mach}/h^2$). A common rule of thumb for the second derivative: $h \sim \varepsilon_{\rm mach}^{1/4} \approx 10^{-4}$.

### Cautions

- Newton's method can converge to a **maximum** or **saddle point** if $f''(x_n) < 0$.
- It requires a **good starting point** — usually obtained from a prior grid search.
- For functions with multiple local minima, it finds the one nearest the starting point.

In [ ]:
# Newton's Method Implementation

def f_prime(f, x, h=1e-5):
    # Central-difference first derivative
    return (f(x + h) - f(x - h)) / (2 * h)

def f_double_prime(f, x, h=1e-4):
    # Central-difference second derivative
    return (f(x + h) - 2 * f(x) + f(x - h)) / h**2

def newton_minimize_1d(f, x0, tol=1e-10, maxiter=50):
    # Newton method for 1-D minimization using numerical derivatives.
    # Returns (x_min, f_min, history_of_iterates).
    x = x0
    history = [x]
    for _ in range(maxiter):
        fp  = f_prime(f, x)
        fpp = f_double_prime(f, x)
        if abs(fpp) < 1e-14:
            print(f"Warning: near-zero second derivative at x={x:.6f}")
            break
        x_new = x - fp / fpp
        history.append(x_new)
        if abs(x_new - x) < tol:
            break
        x = x_new
    return x, f(x), history


# Demo: f(x) = x^4 - 3x^2 + 2x  (two local minima)
def test_func(x):
    return x**4 - 3*x**2 + 2*x

x_arr_t = np.linspace(-2, 2, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x_arr_t, test_func(x_arr_t), 'steelblue')
axes[0].set_title(r'$f(x) = x^4 - 3x^2 + 2x$ — two local minima')
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$f(x)$')

for x0_n, col, lbl in [(-1.5, 'red', '$x_0=-1.5$'), (1.0, 'limegreen', '$x_0=+1.0$')]:
    xm, fm, hist_n = newton_minimize_1d(test_func, x0_n)
    print(f"Start {x0_n:+.1f} -> x* = {xm:.8f},  f* = {fm:.8f},  iters = {len(hist_n)-1}")
    axes[0].plot(xm, fm, 'o', color=col, ms=12, zorder=5,
                 label=f'{lbl} -> $x^*={xm:.4f}$')
    step_sizes = [abs(hist_n[i+1] - hist_n[i]) for i in range(len(hist_n)-1)]
    axes[1].semilogy(step_sizes, 'o-', color=col, ms=6, label=lbl)

axes[0].legend(fontsize=10)
axes[1].set_xlabel('Newton iteration')
axes[1].set_ylabel('Step size $|x_{n+1} - x_n|$')
axes[1].set_title("Newton's Method: Quadratic Convergence")
axes[1].legend()

plt.tight_layout()
plt.show()

### Physics example 8 — Van der Waals gas: spinodal points

The van der Waals equation of state is:

$$\left(P + \frac{a}{V^2}\right)(V - b) = RT$$

Rearranging for pressure at fixed temperature $T$:

$$P(V) = \frac{RT}{V - b} - \frac{a}{V^2}$$

Below the critical temperature, this isotherm has an S-shape with a local maximum and a local minimum — the **spinodal points** — bounding the region of thermodynamic instability. We locate them by finding where $dP/dV = 0$, i.e., by applying Newton's method to the equation $f(V) = -P(V)$ (to turn the max into a min, and vice versa), starting from different initial guesses.

Using analytic derivatives:

$$\frac{dP}{dV} = -\frac{RT}{(V-b)^2} + \frac{2a}{V^3}, \qquad \frac{d^2P}{dV^2} = \frac{2RT}{(V-b)^3} - \frac{6a}{V^4}$$

In [ ]:
# Newton's Method with Analytic Derivatives: Van der Waals Spinodal Points

R_gas = 8.314       # J/(mol K)
# CO2: a = 0.3658 Pa m^6/mol^2, b = 42.9e-6 m^3/mol
a_vdw, b_vdw = 0.3658, 42.9e-6
T_c_vdw = 8 * a_vdw / (27 * R_gas * b_vdw)
print(f"Critical temperature of CO2: T_c = {T_c_vdw:.2f} K")

T_sub = 0.85 * T_c_vdw   # subcritical

def P_vdw(V):
    return R_gas * T_sub / (V - b_vdw) - a_vdw / V**2

def dP_dV(V):
    # Analytic first derivative
    return -R_gas * T_sub / (V - b_vdw)**2 + 2 * a_vdw / V**3

def d2P_dV2(V):
    # Analytic second derivative
    return 2 * R_gas * T_sub / (V - b_vdw)**3 - 6 * a_vdw / V**4

# Newton on dP/dV = 0 using analytic derivatives
def newton_analytic(fp, fpp, x0, tol=1e-12, maxiter=60):
    x = x0; hist = [x]
    for _ in range(maxiter):
        dx = -fp(x) / fpp(x)
        x += dx; hist.append(x)
        if abs(dx) < tol: break
    return x, hist

V_c = 3 * b_vdw
V_spino_max, hist_smax = newton_analytic(dP_dV, d2P_dV2, x0=V_c * 0.6)
V_spino_min, hist_smin = newton_analytic(dP_dV, d2P_dV2, x0=V_c * 2.5)

print(f"Spinodal max:  V = {V_spino_max:.4e} m^3/mol,  P = {P_vdw(V_spino_max)/1e6:.4f} MPa")
print(f"Spinodal min:  V = {V_spino_min:.4e} m^3/mol,  P = {P_vdw(V_spino_min)/1e6:.4f} MPa")

V_scan = np.linspace(b_vdw * 1.05, 6e-4, 2000)
P_scan = P_vdw(V_scan)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(V_scan * 1e6, P_scan / 1e6, 'steelblue')
axes[0].plot(V_spino_max * 1e6, P_vdw(V_spino_max) / 1e6, 'rv', ms=12, zorder=5,
             label=f'Spinodal max  $V={V_spino_max*1e6:.1f}$ cm$^3$/mol')
axes[0].plot(V_spino_min * 1e6, P_vdw(V_spino_min) / 1e6, 'g^', ms=12, zorder=5,
             label=f'Spinodal min  $V={V_spino_min*1e6:.1f}$ cm$^3$/mol')
axes[0].axhline(0, color='gray', lw=1, ls='--')
axes[0].set_xlabel('Molar volume $V$ (cm$^3$/mol)')
axes[0].set_ylabel('Pressure $P$ (MPa)')
axes[0].set_title(f'Van der Waals Isotherm  ($T = {T_sub:.1f}$ K)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(-5, 12)

axes[1].semilogy([abs(h - V_spino_max) for h in hist_smax[:-1]], 'r-o', ms=6, label='Spinodal max')
axes[1].semilogy([abs(h - V_spino_min) for h in hist_smin[:-1]], 'g-^', ms=6, label='Spinodal min')
axes[1].set_xlabel('Newton iteration')
axes[1].set_ylabel('$|V_n - V^*|$')
axes[1].set_title("Newton's Method Convergence (quadratic)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Application: Least-Squares Fitting as Minimization

One of the most common uses of minimization in experimental physics is **curve fitting**. Given data points $(t_i, y_i)$ with measurement uncertainties $\sigma_i$, we find model parameters $\boldsymbol{\theta}$ by minimizing the **chi-squared** statistic:

$$\chi^2(\boldsymbol{\theta}) = \sum_{i=1}^{N} \left(\frac{y_i - f(t_i;\,\boldsymbol{\theta})}{\sigma_i}\right)^2$$

For a single free parameter, this is a 1-D minimization problem solvable by any method above. For multiple parameters, we need multi-dimensional minimization (discussed in the next section).

### Interpreting $\chi^2$

A well-fitted model with $N$ data points and $p$ free parameters should give:

$$\chi^2_{\rm reduced} = \frac{\chi^2}{N - p} \approx 1$$

If $\chi^2_{\rm reduced} \gg 1$, the model is a poor fit. If $\chi^2_{\rm reduced} \ll 1$, the uncertainties were overestimated.

### Physics example 9 — Radioactive decay fitting

Radioactive decay follows $N(t) = N_0\,e^{-\lambda t}$. We generate synthetic data with Poisson noise (realistic for counting experiments) and fit $\lambda$ by minimizing $\chi^2(\lambda)$, treating $N_0$ as an analytically determined function of $\lambda$ for each fixed $\lambda$.

In [ ]:
# Chi-Squared Fitting: Radioactive Decay

rng = np.random.default_rng(42)

N0_true, lam_true = 1000.0, 0.3   # counts, 1/s
t_data = np.linspace(0, 15, 25)
N_true = N0_true * np.exp(-lam_true * t_data)
N_data = rng.poisson(N_true).astype(float)
sigma  = np.sqrt(N_data + 1)   # Poisson: std ~ sqrt(N)

def chi2_decay(lam, return_N0=False):
    # Chi-squared as a function of lambda.
    # For each lambda, N0 is chosen analytically (weighted linear regression).
    model = np.exp(-lam * t_data)
    w     = 1 / sigma**2
    N0_opt = np.sum(w * N_data * model) / np.sum(w * model**2)
    resid  = (N_data - N0_opt * model) / sigma
    chi2_val = np.sum(resid**2)
    if return_N0:
        return chi2_val, N0_opt
    return chi2_val

result_decay = minimize_scalar(chi2_decay, bounds=(0.05, 1.5), method='bounded')
lam_fit = result_decay.x
chi2_min, N0_fit = chi2_decay(lam_fit, return_N0=True)
dof = len(t_data) - 2   # two fitted parameters (N0, lambda)

print(f"True:   lambda = {lam_true:.4f} s^-1,   N0 = {N0_true:.1f}")
print(f"Fitted: lambda = {lam_fit:.4f} s^-1,   N0 = {N0_fit:.1f}")
print(f"chi^2 = {chi2_min:.2f}  (dof={dof},  reduced chi^2 = {chi2_min/dof:.2f})")

lam_scan = np.linspace(0.05, 0.8, 400)
chi2_scan = [chi2_decay(l) for l in lam_scan]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t_smooth3 = np.linspace(0, 15, 300)
axes[0].errorbar(t_data, N_data, yerr=sigma, fmt='ko', ms=5, capsize=3,
                 label='Data (Poisson)', zorder=5)
axes[0].plot(t_smooth3, N0_true * np.exp(-lam_true * t_smooth3), 'g--',
             lw=2, label=f'True  $\\lambda={lam_true}$')
axes[0].plot(t_smooth3, N0_fit * np.exp(-lam_fit * t_smooth3), 'r-',
             lw=2, label=f'Fit   $\\lambda={lam_fit:.4f}$')
axes[0].set_xlabel('Time $t$ (s)')
axes[0].set_ylabel('Counts $N$')
axes[0].set_title('Radioactive Decay — Chi-Squared Fit')
axes[0].legend(fontsize=10)

axes[1].plot(lam_scan, chi2_scan, 'steelblue')
axes[1].axvline(lam_fit,  color='red',   ls='--', label=f'Fit  $\\lambda={lam_fit:.4f}$')
axes[1].axvline(lam_true, color='green', ls=':',  lw=2, label=f'True $\\lambda={lam_true}$')
axes[1].axhline(chi2_min + 1, color='orange', ls='--', lw=1.5,
                label=r'$\chi^2_{\rm min}+1$ (1$\sigma$ bound)')
axes[1].set_xlabel('Decay constant $\\lambda$ (s$^{-1}$)')
axes[1].set_ylabel(r'$\chi^2(\lambda)$')
axes[1].set_title(r'$\chi^2$ Landscape')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Comparing All 1-D Methods

Let us run all methods on the same benchmark function and compare convergence speed directly. We use the Morse potential.

In [ ]:
# Convergence Comparison: All 1-D Methods on Morse Potential

De, a_morse, re = 1.0, 1.5, 1.0

def morse(r):
    return De * (1 - np.exp(-a_morse * (r - re)))**2

EXACT = re
a0, c0, b0 = 0.3, 3.5, 0.9

# Golden-section: track estimate at each step
def gs_tracked(f, a, c, tol=1e-13, maxiter=400):
    phi = (1 + np.sqrt(5)) / 2; resphi = 2 - phi
    x1 = a + resphi*(c-a); x2 = c - resphi*(c-a)
    f1, f2 = f(x1), f(x2)
    est = [(a+c)/2]
    for _ in range(maxiter):
        if f1 < f2:
            c, x2, f2 = x2, x1, f1
            x1 = a + resphi*(c-a); f1 = f(x1)
        else:
            a, x1, f1 = x1, x2, f2
            x2 = c - resphi*(c-a); f2 = f(x2)
        est.append((a+c)/2)
        if c - a < tol: break
    return (a+c)/2, est

_, est_gs_cmp   = gs_tracked(morse, a0, c0)
_, _, est_para_cmp = parabolic_search(morse, a0, b0, c0, tol=1e-13)
_, _, est_newt_cmp = newton_minimize_1d(morse, x0=0.7, tol=1e-13)

err_gs_cmp   = [abs(e - EXACT) + 1e-16 for e in est_gs_cmp]
err_para_cmp = [abs(e - EXACT) + 1e-16 for e in est_para_cmp]
err_newt_cmp = [abs(e - EXACT) + 1e-16 for e in est_newt_cmp]

# Count Brent function evaluations
calls = [0]
def morse_c(r): calls[0] += 1; return morse(r)
res_b = minimize_scalar(morse_c, bounds=(a0, c0), method='bounded', options={'xatol':1e-13})

print(f"Golden-section:  {len(est_gs_cmp):3d} steps,  final error = {err_gs_cmp[-1]:.2e}")
print(f"Parabolic:       {len(est_para_cmp):3d} steps,  final error = {err_para_cmp[-1]:.2e}")
print(f"Newton:          {len(est_newt_cmp):3d} steps,  final error = {err_newt_cmp[-1]:.2e}")
print(f"Brent (scipy):   {calls[0]:3d} f-evals, x* = {res_b.x:.14f}")

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(range(len(err_gs_cmp)),   err_gs_cmp,   'b-o',  ms=4,
            label=f'Golden-section ({len(est_gs_cmp)} iter)')
ax.semilogy(range(len(err_para_cmp)), err_para_cmp, 'g-s',  ms=5,
            label=f'Parabolic ({len(est_para_cmp)} iter)')
ax.semilogy(range(len(err_newt_cmp)), err_newt_cmp, 'r-^',  ms=7,
            label=f"Newton ({len(est_newt_cmp)} iter)")
ax.set_xlabel('Iteration / step')
ax.set_ylabel('$|x_n - x^*|$  (log scale)')
ax.set_title('Convergence Comparison — Morse Potential  ($r_e = 1.0$)')
ax.legend(fontsize=11)
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

---
## Brief Overview: Multi-Dimensional Minimization

Real physics problems often involve many parameters simultaneously — fitting a spectrum with a dozen peaks, optimizing a molecular structure, training a neural network. The extension of 1-D methods to $d$ dimensions introduces new challenges and richer algorithmic ideas.

### Gradient Descent

The simplest extension — move in the direction of steepest descent, $-\nabla f$:

$$\mathbf{x}_{n+1} = \mathbf{x}_n - \alpha\, \nabla f(\mathbf{x}_n)$$

The step size $\alpha$ (the "learning rate") must be chosen carefully. Too large: oscillations or divergence. Too small: extremely slow convergence. Steepest descent converges only linearly and can zig-zag badly in elongated valleys.

### Newton's Method in Multiple Dimensions

Using the **Hessian matrix** $H_{ij} = \partial^2 f / \partial x_i \partial x_j$, Newton's update is:

$$\mathbf{x}_{n+1} = \mathbf{x}_n - \mathbf{H}^{-1}(\mathbf{x}_n)\,\nabla f(\mathbf{x}_n)$$

This converges quadratically near the minimum but computing and inverting $\mathbf{H}$ costs $O(d^3)$ per step — expensive for large $d$.

### Quasi-Newton Methods (BFGS)

**BFGS** (Broyden–Fletcher–Goldfarb–Shanno) builds an approximation $\mathbf{B}_n \approx \mathbf{H}^{-1}$ updated at each step using gradient information, at $O(d^2)$ cost. This achieves superlinear convergence and is the best general-purpose choice for smooth, unconstrained problems.

The update rule is:

$$\mathbf{B}_{n+1} = \mathbf{B}_n + \frac{\mathbf{s}\mathbf{s}^T}{\mathbf{s}^T\mathbf{y}} - \frac{\mathbf{B}_n\mathbf{y}\mathbf{y}^T\mathbf{B}_n}{\mathbf{y}^T\mathbf{B}_n\mathbf{y}}$$

where $\mathbf{s} = \mathbf{x}_{n+1} - \mathbf{x}_n$ and $\mathbf{y} = \nabla f_{n+1} - \nabla f_n$.

### Nelder–Mead (Simplex) Method

A derivative-free method for $d$ dimensions. Maintains a simplex of $d+1$ points and iteratively reflects, expands, or contracts it. Robust and easy to use, but converges only linearly and becomes slow for $d \gtrsim 10$.

### Method comparison

| Method | Derivatives | Cost/step | Convergence | Best for |
|---|---|---|---|---|
| Gradient descent | $\nabla f$ | $O(d)$ | Linear | Large-scale, noisy |
| Nelder-Mead | None | $O(d)$ | Linear | Low $d$, non-smooth |
| BFGS | $\nabla f$ | $O(d^2)$ | Superlinear | Smooth, moderate $d$ |
| Newton | $\nabla f$, $\mathbf{H}$ | $O(d^3)$ | Quadratic | Smooth, small $d$ |
| L-BFGS | $\nabla f$ | $O(dm)$ | Superlinear | Large $d$ |

In [ ]:
# Multi-dimensional minimization with scipy.optimize.minimize
from scipy.optimize import minimize

# Physics example: fit a damped oscillator
#   y(t) = A * exp(-t/tau) * cos(omega0 * t)
# to noisy data, finding A and tau simultaneously.

rng2    = np.random.default_rng(7)
omega0  = 2 * np.pi * 1.0   # 1 Hz
A_true, tau_true = 3.0, 2.5

t_fit  = np.linspace(0, 5, 60)
y_true = A_true * np.exp(-t_fit / tau_true) * np.cos(omega0 * t_fit)
y_data = y_true + rng2.normal(0, 0.2, size=len(t_fit))
sigma2 = 0.2**2

def chi2_2d(params):
    A, tau = params
    if tau <= 0 or A <= 0:
        return 1e10
    model = A * np.exp(-t_fit / tau) * np.cos(omega0 * t_fit)
    return np.sum((y_data - model)**2 / sigma2)

res_bfgs = minimize(chi2_2d, x0=[2.0, 1.5], method='BFGS')
A_fit2, tau_fit2 = res_bfgs.x
print(f"True:   A = {A_true:.4f},  tau = {tau_true:.4f}")
print(f"Fitted: A = {A_fit2:.4f},  tau = {tau_fit2:.4f}")
print(f"chi^2 = {res_bfgs.fun:.2f},  success = {res_bfgs.success}")

# 2-D chi^2 landscape
A_scan2   = np.linspace(1.5, 5.0, 80)
tau_scan2 = np.linspace(0.5, 5.0, 80)
AA, TT    = np.meshgrid(A_scan2, tau_scan2)
CHI2      = np.array([[chi2_2d([a, t]) for a in A_scan2] for t in tau_scan2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t_smooth4 = np.linspace(0, 5, 300)
axes[0].scatter(t_fit, y_data, s=18, color='gray', zorder=3, label='Noisy data')
axes[0].plot(t_smooth4, A_true * np.exp(-t_smooth4/tau_true) * np.cos(omega0*t_smooth4),
             'g--', lw=2, label=f'True  $A={A_true},\\,\\tau={tau_true}$')
axes[0].plot(t_smooth4, A_fit2 * np.exp(-t_smooth4/tau_fit2) * np.cos(omega0*t_smooth4),
             'r-', lw=2, label=f'BFGS  $A={A_fit2:.3f},\\,\\tau={tau_fit2:.3f}$')
axes[0].set_xlabel('Time $t$ (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Damped Oscillator Fit — BFGS')
axes[0].legend(fontsize=10)

cp = axes[1].contourf(AA, TT, CHI2, levels=40, cmap='viridis_r')
fig.colorbar(cp, ax=axes[1], label=r'$\chi^2(A, \tau)$')
axes[1].contour(AA, TT, CHI2, levels=10, colors='white', linewidths=0.5, alpha=0.5)
axes[1].plot(A_true,  tau_true,  'g*', ms=15, zorder=5, label='True parameters')
axes[1].plot(A_fit2,  tau_fit2,  'r^', ms=12, zorder=5, label='BFGS minimum')
axes[1].set_xlabel('Amplitude $A$')
axes[1].set_ylabel(r'Decay time $\tau$ (s)')
axes[1].set_title(r'$\chi^2$ Landscape in 2-D Parameter Space')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## Practical Guide: Choosing a Method

Here is a decision flowchart for tackling minimization problems in practice:

1. **Always start with a plot or grid search.** This reveals the global landscape, locates all local minima, and identifies a good starting bracket. Never apply a local method blindly.

2. **For 1-D minimization:**
   - Reliable black-box: use `scipy.optimize.minimize_scalar(f, bounds=(a,b), method='bounded')` — this runs Brent's method.
   - Need to understand convergence behavior: implement golden-section or parabolic interpolation.
   - Analytic or cheap numerical derivative available: Newton's method converges fastest.

3. **For multi-dimensional minimization:**
   - Smooth, gradient available: `scipy.optimize.minimize(f, x0, method='BFGS')`.
   - Noisy or non-differentiable: `method='Nelder-Mead'`.
   - Very large parameter space ($d \gtrsim 100$): `method='L-BFGS-B'`.
   - Always try multiple starting points to check for multiple local minima.

4. **Quantify uncertainty.** For $\chi^2$ fitting, the $1\sigma$ uncertainty on a parameter corresponds to $\Delta\chi^2 = 1$ around the minimum (one parameter of interest, all others optimized). This is the **profile likelihood** method.

---

## Summary of Methods

| Method | Type | Convergence | Key strength |
|---|---|---|---|
| Grid search | Derivative-free | $O(1/N)$ | Global landscape, always works |
| Golden-section | Derivative-free | Linear | Guaranteed; optimal bracket shrinkage |
| Parabolic interpolation | Derivative-free | Superlinear | Fast near smooth minima |
| Brent's method | Derivative-free | Superlinear (guaranteed) | Best general 1-D method |
| Newton's method | Uses $f'$, $f''$ | Quadratic | Fastest convergence near minimum |
| BFGS | Uses $\nabla f$ | Superlinear | Best general multi-dimensional method |
| Nelder-Mead | Derivative-free | Linear | Robust for non-smooth multi-dim. |

Function minimization is a skill you will return to again and again. The Lennard-Jones pair potential, Fermat's principle, Planck's spectrum, radioactive decay, orbital mechanics, and the van der Waals equation of state are just a few of the physics problems where efficient minimization is essential. Mastering these tools — and understanding when each one is appropriate — is a foundational skill for any computational physicist.